In [1]:
import os, re, json, time, requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()                       # читает src/.env
USDA_API_KEY = os.getenv("USDA_API_KEY")

DATA = "data"
OUT  = "."

print("cwd:", os.getcwd())
print("data/ существует:", os.path.isdir(DATA))
for f in ["epi_r.csv", "full_format_recipes.json", "clean_ingredients.csv"]:
    print(f"  {f}:", os.path.isfile(f"{DATA}/{f}"))

print("\nAPI-ключ загружен:", bool(USDA_API_KEY),
      "| длина:", len(USDA_API_KEY) if USDA_API_KEY else 0)

cwd: /home/ocean127/sber/DS_Bootcamp_Part2/DSB12_Food_nutrition_ID_1577655-Team_TL_pineanne_72024d37_debf_4305-1/src
data/ существует: True
  epi_r.csv: True
  full_format_recipes.json: True
  clean_ingredients.csv: True

API-ключ загружен: True | длина: 40


In [2]:
# =============================================================
# 2. LOAD RAW DATA
# =============================================================
raw = pd.read_csv(f"{DATA}/epi_r.csv")
print("epi_r.csv (до чистки):", raw.shape)
print("  milk в столбцах:  ", "milk" in raw.columns)
print("  jam  в столбцах:  ", "jam"  in raw.columns)
print("  title в столбцах: ", "title" in raw.columns)

# --- 1. Дублируем составные столбцы под канонические имена ---
#     Оригиналы (milk/cream, jam or jelly) остаются в raw — прозрачнее,
#     чем rename. В ml_df попадут только канонические (фильтр в ячейке 3).
if "milk/cream" in raw.columns:
    raw["milk"] = raw["milk/cream"]
elif "milk" not in raw.columns:
    print("⚠️  Нет ни milk, ни milk/cream в raw")

if "jam or jelly" in raw.columns:
    raw["jam"] = raw["jam or jelly"]
elif "jam" not in raw.columns:
    print("⚠️  Нет ни jam, ни jam or jelly в raw")

# --- 2. Убираем составные НЕ-ингредиенты (техники, блюда, диет-теги) ---
NOISE_COMPOSITE = {
    "casserole/gratin", "cobbler/crumble", "condiment/spread",
    "freeze/chill", "grill/barbecue", "low/no sugar",
    "poker/game night", "rosh hashanah/yom kippur",
    "soufflé/meringue", "soup/stew", "stuffing/dressing",
    "wheat/gluten-free",
}
noise_in_raw = [c for c in NOISE_COMPOSITE if c in raw.columns]
if noise_in_raw:
    raw = raw.drop(columns=noise_in_raw)
    print(f"✅ Убраны не-ингредиенты: {len(noise_in_raw)} столбцов")

# --- 3. Проверки ---
print("\nepi_r.csv (после чистки):", raw.shape)
print("  milk в столбцах:  ", "milk" in raw.columns)
print("  jam  в столбцах:  ", "jam"  in raw.columns)
print("  soup/stew убран:  ", "soup/stew" not in raw.columns)

# --- 4. Загрузка полных рецептов ---
with open(f"{DATA}/full_format_recipes.json", encoding="utf-8") as f:
    recipes_json = json.load(f)
print("\nfull_format_recipes.json:", len(recipes_json), "рецептов")

# --- 5. Канонические ингредиенты ---
clean = pd.read_csv(f"{DATA}/clean_ingredients.csv")
print("\nclean_ingredients.csv: shape =", clean.shape)

epi_r.csv (до чистки): (20052, 680)
  milk в столбцах:   False
  jam  в столбцах:   False
  title в столбцах:  True
✅ Убраны не-ингредиенты: 12 столбцов

epi_r.csv (после чистки): (20052, 670)
  milk в столбцах:   True
  jam  в столбцах:   True
  soup/stew убран:   True

full_format_recipes.json: 20130 рецептов

clean_ingredients.csv: shape = (497, 1)


In [3]:
# =============================================================
# 3. ML DATAFRAME
# =============================================================
# 1) "Не-ингредиентные" столбцы — по чек-листу ИХ НЕ ДОЛЖНО БЫТЬ в ml_df
FORBIDDEN = {
    "title", "leftovers", "california", "dominican republic",
    "protein", "low cholesterol", "pan-fry", "fruit", "stew", "braise",
    "rating", "calories", "fat", "sodium", "date", "desc",
    "directions", "ingredients", "categories",
}

# 2) Список настоящих ингредиентов — из clean_ingredients.csv
if "ingredient" in clean.columns:
    ingredients_set = set(clean["ingredient"].astype(str).str.lower().str.strip())
else:
    ingredients_set = set(clean.iloc[:, 0].astype(str).str.lower().str.strip())

# 3) Оставляем: rating + только те столбцы, что есть в clean_ingredients
keep_cols = ["rating"] + [
    c for c in raw.columns
    if c.lower().strip() in ingredients_set       # ← добавил .strip() на всякий случай
    and c not in FORBIDDEN
    and not c.startswith("#")
]

ml_df = raw[keep_cols].copy()
print("ml_df:", ml_df.shape)
print("milk в ml_df:", "milk" in ml_df.columns)
print("jam  в ml_df:", "jam"  in ml_df.columns)

ml_df: (20052, 489)
milk в ml_df: True
jam  в ml_df: True


In [4]:
need_present = ["milk", "jam"]
need_absent  = [
    "title", "leftovers", "california", "dominican republic",
    "protein", "low cholesterol", "pan-fry", "fruit", "stew", "braise",
]

print("=== Должны БЫТЬ ===")
for c in need_present:
    print(f"  {c:25s}: {c in ml_df.columns}")

print("=== Должны ОТСУТСТВОВАТЬ ===")
for c in need_absent:
    print(f"  {c:25s}: {c in ml_df.columns}  (False = OK)")

# Мягкие ассерты — чтобы не падало, но предупреждало
missing = [c for c in need_present if c not in ml_df.columns]
present_forbidden = [c for c in need_absent if c in ml_df.columns]
if missing or present_forbidden:
    print(f"\n⚠️  missing={missing}, лишние={present_forbidden}")

=== Должны БЫТЬ ===
  milk                     : True
  jam                      : True
=== Должны ОТСУТСТВОВАТЬ ===
  title                    : False  (False = OK)
  leftovers                : False  (False = OK)
  california               : False  (False = OK)
  dominican republic       : False  (False = OK)
  protein                  : False  (False = OK)
  low cholesterol          : False  (False = OK)
  pan-fry                  : False  (False = OK)
  fruit                    : False  (False = OK)
  stew                     : False  (False = OK)
  braise                   : False  (False = OK)


Модели регрессии и классификации (партнер А)

In [5]:
USDA_SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"

# FDC nutrientId → (название, единица, суточная норма по FDA)
DAILY_VALUES = {
    1003: ("Protein",            "g",    50.0),
    1004: ("Total Fat",          "g",    78.0),
    1005: ("Total Carbohydrate", "g",   275.0),
    1079: ("Dietary Fiber",      "g",    28.0),
    1258: ("Saturated Fat",      "g",    20.0),
    1253: ("Cholesterol",        "mg",  300.0),
    1093: ("Sodium",             "mg", 2300.0),
    1087: ("Calcium",            "mg", 1300.0),
    1089: ("Iron",               "mg",   18.0),
    1092: ("Potassium",          "mg", 4700.0),
    1162: ("Vitamin C",          "mg",   90.0),
    1106: ("Vitamin A",          "mcg", 900.0),
    1114: ("Vitamin D",          "mcg",  20.0),
}


BAD_STARTS = (
    # десерты/выпечка
    "cake", "crackers", "cookies", "candies", "cereal",
    "snacks", "pasta", "pie", "pizza", "soup", "sausage", "fast food",
    # жиры/масла
    "oil", "shortening", "lard", "margarine",
    # молочка-полуфабрикаты
    "butter,", "cream,", "cheese, cream", "cheese, cottage",
    "cheese food", "cheese spread", "cheese sauce", "cheese dip",
    # сдобный хлеб
    "bread, egg", "bread, stuffing", "bread, cheese", "bread, raisin",
    "bread, banana", "bread, fruit", "bread, cranberry", "bread, nut",
    "bread, garlic", "bread, olive", "bread, brioche", "bread, panettone",
    # прочее
    "beverages", "babyfood",
)


def _usda_query(q, api_key):
    try:
        r = requests.post(
            USDA_SEARCH_URL,
            params={"api_key": api_key},
            json={"query": q, "pageSize": 15,
                  "dataType": ["Foundation", "SR Legacy"]},
            timeout=30,
        )
        if r.status_code != 200:
            return []
        if "application/json" not in r.headers.get("Content-Type", ""):
            return []
        return r.json().get("foods") or []
    except (requests.RequestException, ValueError, json.JSONDecodeError):
        return []


def _pick_best(foods, q_low):
    """Лучший food. НЕ возвращает foods[0] fallback — только None если нет."""
    for f in foods:
        desc = (f.get("description") or "").lower().strip()
        if not desc or desc.startswith(BAD_STARTS):
            continue
        if desc.startswith(q_low):
            return f
    for f in foods:
        desc = (f.get("description") or "").lower().strip()
        if not desc or desc.startswith(BAD_STARTS):
            continue
        if q_low in desc.split(",")[0]:
            return f
    for f in foods:
        desc = (f.get("description") or "").lower().strip()
        if not desc or desc.startswith(BAD_STARTS):
            continue
        if q_low in desc:
            return f
    return None


def fetch_food_nutrients(query, api_key, retries=2):
    if not api_key:
        return {}
    q_low = query.lower().strip()
    candidates = [
        q_low,
        f"{q_low} raw",
        f"{q_low}s, raw",
        f"{q_low}, raw",
    ]
    for attempt in range(retries):
        for q in candidates:
            foods = _usda_query(q, api_key)
            if not foods:
                continue
            best = _pick_best(foods, q_low)
            if best is None:
                continue
            out = {}
            for n in best.get("foodNutrients", []):
                nid = n.get("nutrientId")
                val = n.get("value")
                if val is None:
                    val = n.get("amount")
                if nid is None or val is None:
                    continue
                try:
                    out[int(nid)] = float(val)
                except (TypeError, ValueError):
                    continue
            if out:
                return out
        time.sleep(1)
    return {}


# Смоук-тест: три ингредиента, которые раньше матчились плохо
if USDA_API_KEY:
    print("Смоук-тест (protein g / fat g на 100g — ожидания в скобках):")
    for ing, expected in [
        ("avocado", "~2 / ~15"),
        ("bread",   "~9 / ~4"),
        ("cheese",  "~25 / ~33"),
    ]:
        d = fetch_food_nutrients(ing, USDA_API_KEY)
        if d:
            print(f"  {ing:8s}: Protein={d.get(1003, 0):.1f}g  "
                  f"Fat={d.get(1004, 0):.1f}g    (ожидаем {expected})")
        else:
            print(f"  {ing:8s}: пусто")

Смоук-тест (protein g / fat g на 100g — ожидания в скобках):
  avocado : Protein=2.0g  Fat=15.4g    (ожидаем ~2 / ~15)
  bread   : Protein=7.0g  Fat=5.3g    (ожидаем ~9 / ~4)
  cheese  : Protein=21.4g  Fat=28.7g    (ожидаем ~25 / ~33)


In [6]:
CACHE_PATH = "usda_raw_cache.json"

ingredient_cols = [c for c in ml_df.columns if c != "rating"]
print("Ингредиентов в очереди:", len(ingredient_cols))

if os.path.isfile(CACHE_PATH):
    with open(CACHE_PATH, encoding="utf-8") as f:
        _raw_cache = json.load(f)
    # ⚠️ JSON читает ключи как строки — конвертируем в int
    nutrients_raw = {
        ing: {int(k): float(v) for k, v in (nutr or {}).items()}
        for ing, nutr in _raw_cache.items()
    }
    print(f"Кэш загружен: {len(nutrients_raw)} ингредиентов (ключи → int)")
else:
    nutrients_raw = {}
    print("Кэша нет — стартуем с нуля")

missing = [ing for ing in ingredient_cols if ing not in nutrients_raw]
if not missing:
    print("✅ Все ингредиенты уже в кэше — API не нужен")
else:
    print(f"В кэше нет {len(missing)} ингредиентов")
    if not USDA_API_KEY:
        print("⚠️  API-ключ не задан — пропускаем недостающие (для сдачи это OK)")
    else:
        for i, ing in enumerate(missing, 1):
            nutrients_raw[ing] = fetch_food_nutrients(ing, USDA_API_KEY)
            if i % 25 == 0:
                print(f"  {i}/{len(missing)}  последний: {ing}")
                with open(CACHE_PATH, "w", encoding="utf-8") as f:
                    json.dump(nutrients_raw, f, ensure_ascii=False)
                time.sleep(0.1)
        with open(CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(nutrients_raw, f, ensure_ascii=False)
        print(f"Кэш обновлён до {len(nutrients_raw)} ингредиентов")

found = sum(1 for v in nutrients_raw.values() if v)
print(f"Итого: {len(nutrients_raw)} ингредиентов, из них с нутриентами: {found}")

# Санити: проверим, что в кэше int-ключи 4-значные
sample = nutrients_raw.get("almond", {})
print("\nКонтроль almond:")
print("  [1003] protein:", sample.get(1003))
print("  [1087] calcium:", sample.get(1087))
print("  [203]  старый SR:", sample.get(203), "(должно быть None)")

Ингредиентов в очереди: 488
Кэш загружен: 496 ингредиентов (ключи → int)
✅ Все ингредиенты уже в кэше — API не нужен
Итого: 496 ингредиентов, из них с нутриентами: 418

Контроль almond:
  [1003] protein: 26.2
  [1087] calcium: 232.0
  [203]  старый SR: None (должно быть None)


In [7]:
NUTRITION_CSV = "nutrition_facts.csv"

rows = []
for ing, nutr in nutrients_raw.items():
    if not nutr:
        continue
    row = {"ingredient": ing}
    for nid, (name, unit, daily) in DAILY_VALUES.items():
        value = nutr.get(nid, 0.0) or 0.0
        row[name] = round(value / daily * 100, 2) if daily else 0.0
    rows.append(row)

nutrition_df = pd.DataFrame(rows).set_index("ingredient")
print("nutrition_facts shape:", nutrition_df.shape)
print(nutrition_df.head(3))

# --- обязательные проверки ---
assert nutrition_df.shape[0] > 50, f"Мало ингредиентов: {nutrition_df.shape[0]}"
assert nutrition_df.shape[1] >= 3,  f"Мало нутриентов: {nutrition_df.shape[1]}"
assert nutrition_df["Protein"].max() > 1.0, "Все нули — проверь DAILY_VALUES/кэш"

nutrition_df.to_csv(NUTRITION_CSV)
print(f"✅ Сохранено: {NUTRITION_CSV}")

# --- контрольные значения ---
print("\nКонтрольные значения (должны быть ненулевые):")
for ing in ["almond", "bread", "avocado", "cheese", "garlic"]:
    if ing in nutrition_df.index:
        r = nutrition_df.loc[ing]
        print(f"  {ing:10s}  Protein={r['Protein']:6.2f}  Fat={r['Total Fat']:6.2f}  "
              f"Carbs={r['Total Carbohydrate']:6.2f}  Ca={r['Calcium']:6.2f}")

nutrition_facts shape: (418, 13)
                      Protein  Total Fat  Total Carbohydrate  Dietary Fiber  \
ingredient                                                                    
3-ingredient recipes     1.72       0.37                1.57            0.0   
30 days of groceries    28.80      38.46                0.00            0.0   
alaska                  34.60       1.27                0.02            0.0   

                      Saturated Fat  Cholesterol  Sodium  Calcium  Iron  \
ingredient                                                                
3-ingredient recipes           0.00          0.0   10.26     0.75  1.66   
30 days of groceries          59.00         26.0    2.87     1.85  9.11   
alaska                         1.19         20.0    5.00     3.11  1.33   

                      Potassium  Vitamin C  Vitamin A  Vitamin D  
ingredient                                                        
3-ingredient recipes       4.21      55.33       0.00        

In [8]:
sample_keys = set()
for r in recipes_json[:50]:
    sample_keys.update(r.keys())
print("Ключи:", sorted(sample_keys))

id_like = [k for k in sample_keys
           if any(x in k.lower() for x in ["id", "url", "link", "slug"])]
print("Похожие на ID:", id_like)
print("Примеры:", {k: recipes_json[0].get(k) for k in id_like})

Ключи: ['calories', 'categories', 'date', 'desc', 'directions', 'fat', 'ingredients', 'protein', 'rating', 'sodium', 'title']
Похожие на ID: []
Примеры: {}


In [9]:
import re

EPI_NON_INGR = {
    "rating", "calories", "protein", "fat", "sodium",
    "title", "date", "desc", "directions", "ingredients", "categories",
    "leftovers", "california", "dominican republic", "low cholesterol",
    "pan-fry", "fruit", "stew", "braise",
}
canon_sorted = sorted(
    [c.lower() for c in raw.columns
     if c.lower() not in EPI_NON_INGR and not c.startswith("#")],
    key=len, reverse=True
)
print("canon_sorted:", len(canon_sorted),
      "| milk:", "milk" in canon_sorted,
      "| jam:",  "jam"  in canon_sorted)


def normalize_ingredients(raw_list) -> set:
    if raw_list is None:
        return set()
    if isinstance(raw_list, float) and pd.isna(raw_list):
        return set()
    parts = re.split(r"[|,]", raw_list) if isinstance(raw_list, str) else list(raw_list)
    text = " " + " ".join(str(x).lower() for x in parts) + " "
    text = re.sub(r"[^a-zа-яё\s-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return {c for c in canon_sorted if " " + c + " " in text}


def make_url(title: str, rid=None) -> str:
    slug = re.sub(r"[^\w\s-]", "", title.lower().strip())
    slug = re.sub(r"[\s_]+", "-", slug)
    slug = re.sub(r"-+", "-", slug).strip("-")
    if rid:
        return f"https://www.epicurious.com/recipes/food/views/{slug}-{rid}"
    return f"https://www.epicurious.com/recipes/food/views/{slug}"


rows = []
for r in recipes_json:
    title  = (r.get("title") or "").strip()
    rating = r.get("rating")
    ings   = r.get("ingredients") or []
    if not title or rating is None or not ings:
        continue

    ingredients_joined = "|".join(str(i).strip().lower() for i in ings)
    norm_set = normalize_ingredients(ingredients_joined)
    if not norm_set:
        continue

    rows.append({
        "title":       title,
        "rating":      float(rating),
        "ingredients": ingredients_joined,
        "url":         make_url(title, r.get("id")),
        "norm_set":    "|".join(sorted(norm_set)),
    })

sim_df = pd.DataFrame(rows)
print("sim_df:", sim_df.shape)
print("пустых url:", sim_df["url"].isna().sum())
print("пустых norm_set:", (sim_df["norm_set"] == "").sum())

canon_sorted: 653 | milk: True | jam: True
sim_df: (19908, 5)
пустых url: 0
пустых norm_set: 0


In [10]:
def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def top_similar(query_ingredients, sim_df, top_n=3, min_score=0.4):
    q = normalize_ingredients(query_ingredients)
    if not q:
        return pd.DataFrame()

    scores = sim_df["norm_set"].apply(
        lambda s: jaccard(q, set(s.split("|")) if s else set())
    )
    best = scores.sort_values(ascending=False).head(top_n)

    if best.iloc[0] < min_score:
        return pd.DataFrame()      # «похожих рецептов нет»

    return sim_df.loc[best.index, ["title", "rating", "url"]].assign(score=best.values)


print("=== КОРОТКИЙ набор (должен найтись) ===")
print(top_similar(["bread", "avocado", "cheese", "garlic"], sim_df).to_string())

print("\n=== ДЛИННЫЙ набор (должен вернуть пусто) ===")
print(top_similar(
    ["bread", "avocado", "cheese", "garlic", "saffron", "gin",
     "chile pepper", "banana", "vodka", "wok", "yogurt", "zucchini"],
    sim_df,
).to_string())

=== КОРОТКИЙ набор (должен найтись) ===
                                      title  rating                                                                                  url  score
1938  Sauteed Mushrooms with Melted Stilton   4.375  https://www.epicurious.com/recipes/food/views/sauteed-mushrooms-with-melted-stilton    0.6
3068                  Avocado Egg-in-a-Hole   3.750                  https://www.epicurious.com/recipes/food/views/avocado-egg-in-a-hole    0.5
8113      Ripe Olive and Goat Cheese Toasts   4.375      https://www.epicurious.com/recipes/food/views/ripe-olive-and-goat-cheese-toasts    0.5

=== ДЛИННЫЙ набор (должен вернуть пусто) ===
Empty DataFrame
Columns: []
Index: []


In [11]:
assert sim_df["url"].isna().sum() == 0, "Есть NaN в url"
assert (sim_df["url"].str.strip() != "").all(), "Есть пустые url"
assert sim_df["norm_set"].ne("").all(), "Есть пустые norm_set"

sim_df.to_csv(f"{OUT}/similar_recipes.csv", index=False)
print("Сохранено: similar_recipes.csv, строк:", len(sim_df))
print(sim_df[["title", "rating", "url"]].head(3).to_string())

Сохранено: similar_recipes.csv, строк: 19908
                                        title  rating                                                                                       url
0              Lentil, Apple, and Turkey Wrap   2.500                https://www.epicurious.com/recipes/food/views/lentil-apple-and-turkey-wrap
1  Boudin Blanc Terrine with Red Onion Confit   4.375  https://www.epicurious.com/recipes/food/views/boudin-blanc-terrine-with-red-onion-confit
2                Potato and Fennel Soup Hodge   3.750                https://www.epicurious.com/recipes/food/views/potato-and-fennel-soup-hodge


In [12]:
for f in ["nutrition_facts.csv", "similar_recipes.csv"]:
    print(f"{f:25s} exists: {os.path.isfile(f'{OUT}/{f}')}")

nf = pd.read_csv(f"{OUT}/nutrition_facts.csv", index_col=0)
print("\nnutrition_facts:", nf.shape, "| столбцы:", nf.columns.tolist())
print("almond:\n", nf.loc["almond"].to_string())

nutrition_facts.csv       exists: True
similar_recipes.csv       exists: True

nutrition_facts: (418, 13) | столбцы: ['Protein', 'Total Fat', 'Total Carbohydrate', 'Dietary Fiber', 'Saturated Fat', 'Cholesterol', 'Sodium', 'Calcium', 'Iron', 'Potassium', 'Vitamin C', 'Vitamin A', 'Vitamin D']
almond:
 Protein               52.40
Total Fat             64.36
Total Carbohydrate     5.89
Dietary Fiber         33.11
Saturated Fat          0.00
Cholesterol            0.00
Sodium                 0.04
Calcium               17.85
Iron                  17.89
Potassium             14.19
Vitamin C              0.00
Vitamin A              0.00
Vitamin D              0.00


In [13]:
from recipes import NutritionFacts, SimilarRecipes

nf = NutritionFacts()
sr = SimilarRecipes()

print("=== NutritionFacts ===")
print("almond в базе:", nf.has("almond"))
print("факты для almond:", nf.get_facts(["almond"]))
print("missing для ['a','b','c','garlic']:", nf.missing(["a", "b", "c", "garlic"]))

print("\n=== SimilarRecipes ===")
short = sr.find_similar(["bread", "avocado", "cheese", "garlic"])
print("короткий набор →", len(short), "рецептов")
for r in short:
    print(f"  {r['title']} (rating {r['rating']}) → {r['url']}")

long = sr.find_similar([
    "bread", "avocado", "cheese", "garlic", "saffron", "gin",
    "chile pepper", "banana", "vodka", "wok", "yogurt", "zucchini"
])
print("\nдлинный набор →", len(long), "рецептов (должно быть 0)")

=== NutritionFacts ===
almond в базе: True
факты для almond: {'almond': {'Protein': 52.4, 'Total Fat': 64.36, 'Total Carbohydrate': 5.89, 'Dietary Fiber': 33.11, 'Saturated Fat': 0.0, 'Cholesterol': 0.0, 'Sodium': 0.04, 'Calcium': 17.85, 'Iron': 17.89, 'Potassium': 14.19, 'Vitamin C': 0.0, 'Vitamin A': 0.0, 'Vitamin D': 0.0}}
missing для ['a','b','c','garlic']: ['a', 'b', 'c']

=== SimilarRecipes ===
короткий набор → 3 рецептов
  Sauteed Mushrooms with Melted Stilton (rating 4.375) → https://www.epicurious.com/recipes/food/views/sauteed-mushrooms-with-melted-stilton
  Avocado Egg-in-a-Hole (rating 3.75) → https://www.epicurious.com/recipes/food/views/avocado-egg-in-a-hole
  Ripe Olive and Goat Cheese Toasts (rating 4.375) → https://www.epicurious.com/recipes/food/views/ripe-olive-and-goat-cheese-toasts

длинный набор → 0 рецептов (должно быть 0)


In [14]:
# =============================================================
# ДИАГНОСТИКА: проверка ноутбука по чек-листу
# =============================================================
import os, json
import pandas as pd

print("=" * 60)
print("1. ПЕРЕМЕННЫЕ В ПАМЯТИ")
print("=" * 60)
for name in ["raw", "clean", "recipes_json", "ml_df", "USDA_API_KEY",
             "nutrients_raw", "nutrition_df", "sim_df"]:
    status = "✅ есть" if name in dir() else "❌ НЕТ (запусти ячейку выше)"
    print(f"  {name:20s}: {status}")

print("\n" + "=" * 60)
print("2. ИСХОДНЫЕ ФАЙЛЫ")
print("=" * 60)
for f in ["data/epi_r.csv", "data/full_format_recipes.json",
          "data/clean_ingredients.csv"]:
    print(f"  {f}: {'✅' if os.path.isfile(f) else '❌'}")

print("\n" + "=" * 60)
print("3. ml_df — КЛЮЧЕВОЙ ДАТАФРЕЙМ ДЛЯ ML")
print("=" * 60)
if "ml_df" in dir():
    print(f"  Размер: {ml_df.shape}")

    print("\n  Должно БЫТЬ (prep_points):")
    for c in ["milk", "jam"]:
        ok = c in ml_df.columns
        print(f"    {c:25s}: {'✅ есть' if ok else '❌ НЕТ → теряем балл'}")

    print("\n  Должно ОТСУТСТВОВАТЬ:")
    absent = ["title", "leftovers", "california", "dominican republic",
              "protein", "low cholesterol", "pan-fry", "fruit", "stew", "braise"]
    for c in absent:
        ok = c not in ml_df.columns
        print(f"    {c:25s}: {'✅ нет' if ok else '❌ ЕСТЬ → теряем балл'}")
else:
    print("  ❌ ml_df нет — ячейка 3 не запущена")

print("\n" + "=" * 60)
print("4. ЕСТЬ ЛИ milk/jam В ИСХОДНИКАХ")
print("=" * 60)
if "raw" in dir() and "ingredients_set" in dir():
    for w in ["milk", "jam"]:
        in_raw = w in raw.columns
        in_clean = w in ingredients_set
        print(f"  {w}: в raw (epi_r.csv) = {'✅' if in_raw else '❌ НЕТ'}, "
              f"в clean_ingredients = {'✅' if in_clean else '❌ НЕТ'}")
    if not all(w in raw.columns for w in ["milk", "jam"]):
        print("\n  ⚠️  В epi_r.csv НЕТ milk/jam → это НЕ код, это файл")
        print("     Нужно перекачать epi_r.csv (задача партнёра A)")

print("\n" + "=" * 60)
print("5. АРТЕФАКТЫ (CSV)")
print("=" * 60)
for f in ["nutrition_facts.csv", "similar_recipes.csv"]:
    exists = os.path.isfile(f)
    size = os.path.getsize(f) / 1024 if exists else 0
    print(f"  {f:25s}: {'✅' if exists else '❌'}  ({size:.1f} KB)")

print("\n" + "=" * 60)
print("6. recipes.py")
print("=" * 60)
if os.path.isfile("recipes.py"):
    try:
        from recipes import NutritionFacts, SimilarRecipes
        nf = NutritionFacts()
        sr = SimilarRecipes()
        short = sr.find_similar(["bread", "avocado", "cheese", "garlic"])
        long_ = sr.find_similar(["bread","avocado","cheese","garlic","saffron","gin",
                                  "chile pepper","banana","vodka","wok","yogurt","zucchini"])
        print(f"  NutritionFacts: ✅ (строк в базе: {len(nf.df)})")
        print(f"  SimilarRecipes: ✅ (короткий набор → {len(short)}, длинный → {len(long_)})")
        if len(short) == 3 and len(long_) == 0:
            print("  ✅ Поведение по чек-листу верное")
        else:
            print("  ⚠️  Поведение не совпадает с чек-листом")
    except Exception as e:
        print(f"  ❌ Ошибка импорта: {e}")
else:
    print("  ❌ recipes.py не найден")

print("\n" + "=" * 60)
print("7. ЕСТЬ ЛИ `...` В КОДЕ ЯЧЕЕК")
print("=" * 60)
try:
    with open("recipes.ipynb", encoding="utf-8") as f:
        nb = json.load(f)
    issues = []
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        for ln, line in enumerate(src.split("\n"), 1):
            code = line.split("#")[0]
            if "..." in code:
                issues.append((i, ln, line.strip()[:60]))
    if issues:
        print(f"  ⚠️  Найдено {len(issues)} строк с `...` в коде:")
        for i, ln, txt in issues:
            print(f"    ячейка #{i}, строка {ln}: {txt}")
    else:
        print("  ✅ Чисто, `...` только в комментариях")
except Exception as e:
    print(f"  Не удалось проверить: {e}")

print("\n" + "=" * 60)
print("ГОТОВО. Если везде ✅ — ноутбук в порядке.")
print("=" * 60)

1. ПЕРЕМЕННЫЕ В ПАМЯТИ
  raw                 : ✅ есть
  clean               : ✅ есть
  recipes_json        : ✅ есть
  ml_df               : ✅ есть
  USDA_API_KEY        : ✅ есть
  nutrients_raw       : ✅ есть
  nutrition_df        : ✅ есть
  sim_df              : ✅ есть

2. ИСХОДНЫЕ ФАЙЛЫ
  data/epi_r.csv: ✅
  data/full_format_recipes.json: ✅
  data/clean_ingredients.csv: ✅

3. ml_df — КЛЮЧЕВОЙ ДАТАФРЕЙМ ДЛЯ ML
  Размер: (20052, 489)

  Должно БЫТЬ (prep_points):
    milk                     : ✅ есть
    jam                      : ✅ есть

  Должно ОТСУТСТВОВАТЬ:
    title                    : ✅ нет
    leftovers                : ✅ нет
    california               : ✅ нет
    dominican republic       : ✅ нет
    protein                  : ✅ нет
    low cholesterol          : ✅ нет
    pan-fry                  : ✅ нет
    fruit                    : ✅ нет
    stew                     : ✅ нет
    braise                   : ✅ нет

4. ЕСТЬ ЛИ milk/jam В ИСХОДНИКАХ
  milk: в raw (epi_r.csv) =